In [2]:
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "data"
# step 1 : loading the data
sales = pd.read_csv(f"{DATA_DIR}/sales.csv")
inventory = pd.read_csv(f"{DATA_DIR}/inventory.csv")
finance = pd.read_csv(f"{DATA_DIR}/finance.csv")
vendors = pd.read_csv(f"{DATA_DIR}/vendors.csv")


# basic stats 
print("sales:    ", sales.shape)
print("inventory:", inventory.shape)
print("finance:  ", finance.shape)
print("vendors:  ", vendors.shape)


sales:     (9157, 9)
inventory: (100, 8)
finance:   (5000, 6)
vendors:   (20, 13)


In [3]:
sales.head()

,date,sku,qty,revenue,region,temperature,rainfall,humidity,weather_condition
0,2022-10-03,SKU060,1,3953.37,North,16.7,40.9,67.0,Rainy
1,2020-06-22,SKU058,9,18397.09,South,24.4,57.4,82.0,Rainy
2,2020-05-27,SKU086,5,88407.28,North,20.2,352.8,75.9,Thunderstorm
3,2021-04-27,SKU044,9,235192.77,West,20.6,64.6,66.5,Rainy
4,2020-04-01,SKU081,14,3525.75,North,44.1,1.2,30.6,Sunny


In [4]:
inventory.head(2)

,sku,name,category,region,qty,reorder_threshold,unit_cost,vendor_id
0,SKU001,Mixer Grinder,Kitchen Appliances,North,41,16,3500,V001
1,SKU002,Pressure Cooker,Kitchen Appliances,North,55,30,2500,V018


In [5]:
# merge the columns from inventory ( category, name, region as home_region)
inventory.rename(columns={'region': 'home_region'}, inplace=True)
df = pd.merge(sales, inventory[['sku', 'category', 'name', 'home_region']], on='sku', how='left')

df.head(2)

,date,sku,qty,revenue,region,temperature,rainfall,humidity,weather_condition,category,name,home_region
0,2022-10-03,SKU060,1,3953.37,North,16.7,40.9,67.0,Rainy,Personal Care,Epilator,South
1,2020-06-22,SKU058,9,18397.09,South,24.4,57.4,82.0,Rainy,Personal Care,Hair Trimmer,Central


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9157 entries, 0 to 9156
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   date               9157 non-null   object 
 1   sku                9157 non-null   object 
 2   qty                9157 non-null   int64  
 3   revenue            9157 non-null   float64
 4   region             9157 non-null   object 
 5   temperature        9157 non-null   object 
 6   rainfall           9157 non-null   float64
 7   humidity           9157 non-null   float64
 8   weather_condition  9157 non-null   object 
 9   category           9157 non-null   object 
 10  name               9157 non-null   object 
 11  home_region        9157 non-null   object 
dtypes: float64(3), int64(1), object(8)
memory usage: 858.6+ KB


In [7]:
before = len(df)
df["temperature"] = pd.to_numeric(df["temperature"], errors="coerce")
n_bad = df["temperature"].isna().sum()
print(f"{n_bad} row(s) with non-numeric temperature -> dropping")
df = df.dropna(subset=["temperature"]).reset_index(drop=True)

1 row(s) with non-numeric temperature -> dropping


In [8]:
df["date"] = pd.to_datetime(df["date"])

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9156 entries, 0 to 9155
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               9156 non-null   datetime64[ns]
 1   sku                9156 non-null   object        
 2   qty                9156 non-null   int64         
 3   revenue            9156 non-null   float64       
 4   region             9156 non-null   object        
 5   temperature        9156 non-null   float64       
 6   rainfall           9156 non-null   float64       
 7   humidity           9156 non-null   float64       
 8   weather_condition  9156 non-null   object        
 9   category           9156 non-null   object        
 10  name               9156 non-null   object        
 11  home_region        9156 non-null   object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(6)
memory usage: 858.5+ KB


In [10]:
# feature engineering 
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

In [11]:
df.head(2)

,date,sku,qty,revenue,region,temperature,rainfall,humidity,weather_condition,category,name,home_region,month,day_of_week,is_weekend
0,2022-10-03,SKU060,1,3953.37,North,16.7,40.9,67.0,Rainy,Personal Care,Epilator,South,10,0,0
1,2020-06-22,SKU058,9,18397.09,South,24.4,57.4,82.0,Rainy,Personal Care,Hair Trimmer,Central,6,0,0


In [13]:
df["day_of_week"].value_counts()

day_of_week
2    1326
1    1325
5    1306
0    1305
3    1305
4    1304
6    1285
Name: count, dtype: int64

In [14]:
from sklearn.preprocessing import LabelEncoder

encoders = {}
for col in ["sku", "category", "region", "weather_condition"]:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    encoders[col] = le


In [15]:
feature_cols = [
    "sku_enc", "category_enc", "region_enc", "weather_condition_enc",
    "temperature", "rainfall", "humidity",
    "month", "day_of_week", "is_weekend",
]

target_col = "qty"

X = df[feature_cols]
y = df[target_col]


In [16]:
df_sorted = df.sort_values("date").reset_index(drop=True)

In [17]:
# step 3 : split the data into 2 category --> time based spliting
split_idx = int(len(df_sorted) * 0.85)
split_date = df_sorted.loc[split_idx, "date"]

train_df = df_sorted[df_sorted["date"] < split_date]
test_df = df_sorted[df_sorted["date"] >= split_date]

In [18]:
train_df.shape,test_df.shape

((7778, 19), (1378, 19))

In [20]:
X_train, y_train = train_df[feature_cols], train_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

In [21]:
pip install xgboost

  Using cached xgboost-3.4.1-py3-none-macosx_12_0_arm64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-macosx_12_0_arm64.whl (2.4 MB)

[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [22]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, mean_absolute_error
import pickle


In [23]:
model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

In [24]:
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=True,
)

[0]	validation_0-rmse:4.85638
[1]	validation_0-rmse:4.84684
[2]	validation_0-rmse:4.84168
[3]	validation_0-rmse:4.83068
[4]	validation_0-rmse:4.82484
[5]	validation_0-rmse:4.81936
[6]	validation_0-rmse:4.81571
[7]	validation_0-rmse:4.81008
[8]	validation_0-rmse:4.80198
[9]	validation_0-rmse:4.79758
[10]	validation_0-rmse:4.79215
[11]	validation_0-rmse:4.78933
[12]	validation_0-rmse:4.78472
[13]	validation_0-rmse:4.78130
[14]	validation_0-rmse:4.78050
[15]	validation_0-rmse:4.77683
[16]	validation_0-rmse:4.77306
[17]	validation_0-rmse:4.77074
[18]	validation_0-rmse:4.76806
[19]	validation_0-rmse:4.76751
[20]	validation_0-rmse:4.76086
[21]	validation_0-rmse:4.75770
[22]	validation_0-rmse:4.75890
[23]	validation_0-rmse:4.75820
[24]	validation_0-rmse:4.75652
[25]	validation_0-rmse:4.75423
[26]	validation_0-rmse:4.75450
[27]	validation_0-rmse:4.75388
[28]	validation_0-rmse:4.75256
[29]	validation_0-rmse:4.75140
[30]	validation_0-rmse:4.75322
[31]	validation_0-rmse:4.75060
[32]	validation_0-

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [27]:
y_pred = model.predict(X_test)


In [25]:
X_test

,sku_enc,category_enc,region_enc,weather_condition_enc,temperature,rainfall,humidity,month,day_of_week,is_weekend
7778,67,4,2,0,29.5,0.0,49.5,2,1,0
7779,82,2,2,1,10.4,8.1,74.0,2,1,0
7780,31,1,4,4,17.6,135.4,82.5,2,1,0
7781,46,0,0,1,22.5,0.0,53.0,2,1,0
7782,88,0,4,0,30.3,0.8,35.1,2,1,0
...,...,...,...,...,...,...,...,...,...,...
9151,49,0,0,1,25.2,13.1,60.6,10,2,0
9152,83,3,3,7,41.4,0.0,43.4,10,2,0
9153,74,6,4,6,19.5,64.0,72.2,10,3,0
9154,36,1,2,0,34.6,3.3,37.2,10,3,0


In [28]:
# save this model
artifact = {
    "model": model,
    "encoders": encoders,
    "feature_cols": feature_cols,
}

with open("forecast_model.pkl", "wb") as f:
    pickle.dump(artifact, f)